In [14]:
# Created on Oct 10, 2025
# Created by: Afeena
# Purpose: This file answers the business questions for the user slice

import pandas as pd
import numpy as np
import os as os
# import sys
import statsmodels.formula.api as smf
from itertools import product


# File Paths
TimeSeveritySlicePath = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\04Regression\\TimeSeveritySlice.csv"
destinationFolder = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\04Regression"

# Read the CSV file
dfTime= pd.read_csv(TimeSeveritySlicePath, low_memory=False)

# Make column for SevereInjury:Fatal/Major = 1, Minor/Minimal = 0
dfTime['SevereInjury'] = np.where(dfTime['SeverityOfInjury'].isin(['Fatal', 'Major']), 1, 0)

# print(df['SeverityOfInjury'].value_counts())
# print(df['SevereInjury'].value_counts())

# identify dummy variable columns
peakCols = [c for c in dfTime.columns if c.startswith('TimePeakOffPeak_')]
seasonCols = [c for c in dfTime.columns if c.startswith('TimeSeason_')]
timeRangeCols = [c for c in dfTime.columns if c.startswith('TimeRange_')]
monthCols = [c for c in dfTime.columns if c.startswith('TimeMonth_')]
hourCols = [c for c in dfTime.columns if c.startswith('TimeHour_')]

# baseline have already been dropped in Wrangler.ipynb

print ("Time peak/off-peak dummies:", peakCols[:5],"...total:", len(peakCols))
print ("Time season dummies:", seasonCols[:5],"...total:", len(seasonCols))
print ("Time range dummies:", timeRangeCols[:5],"...total:", len(timeRangeCols))
print ("Time month dummies:", monthCols[:5],"...total:", len(monthCols))
print ("Time hour dummies:", hourCols[:5],"...total:", len(hourCols))

# ModelA: How does time peak/off-peak influence the probability of severe injuries?
formulaA = 'SevereInjury ~ ' + ' + '.join(peakCols)
modelA = smf.logit(formula=formulaA, data=dfTime).fit()
print("\n Model A: Time Peak/Off-Peak vs Severe Injuries \n")
print(modelA.summary())

# Save model summary to csv file
with open(os.path.join(destinationFolder, "TimeModelA_Stats.csv"), 'w') as f:
    f.write(str(modelA.summary()))

# ModelB: how does season influence the probability of severe injuries?
formulaB = 'SevereInjury ~ ' + ' + '.join(seasonCols)
modelB = smf.logit(formula=formulaB, data=dfTime).fit()
print("\n Model B: Time Season vs Severe Injuries \n")
print(modelB.summary())

# Save model summary to csv file
with open(os.path.join(destinationFolder, "TimeModelB_Stats.csv"), 'w') as f:
    f.write(str(modelB.summary()))

# ModelC: how does month influence the probability of severe injuries?
formulaC = 'SevereInjury ~ ' + ' + '.join(monthCols)
modelC = smf.logit(formula=formulaC, data=dfTime).fit()
print("\n Model C: Time Month vs Severe Injuries \n")
print(modelC.summary())

# Save model summary to csv file
with open(os.path.join(destinationFolder, "TimeModelC_Stats.csv"), 'w') as f:
    f.write(str(modelC.summary()))

# ModelD: how does hour influence the probability of severe injuries?
formulaD = 'SevereInjury ~ ' + ' + '.join(hourCols)
modelD = smf.logit(formula=formulaD, data=dfTime).fit()
print("\n Model D: Time Hour vs Severe Injuries \n")
print(modelD.summary())

# Save model summary to csv file
with open(os.path.join(destinationFolder, "TimeModelD_Stats.csv"), 'w') as f:
    f.write(str(modelD.summary()))


# ModelE: how does time range influence the probability of severe injuries?
formulaE = 'SevereInjury ~ ' + ' + '.join(timeRangeCols)
modelE = smf.logit(formula=formulaE, data=dfTime).fit()
print("\n Model E: Time Range vs Severe Injuries \n")
print(modelE.summary())

# Save model summary to csv file
with open(os.path.join(destinationFolder, "TimeModelE_Stats.csv"), 'w') as f:
    f.write(str(modelE.summary()))

# Function for Odds Ratios and Confidence Intervals
# the Odd Ratios and Summary function above will give the same results.
def odds_ratios_and_ci(whichModel):
    params = whichModel.params
    conf = whichModel.conf_int()
    out = pd.DataFrame({
        # Odds Ratios
        "OR": np.exp(params),
        # Confidence Intervals
        "2.5%": np.exp(conf[0]),
        "97.5%": np.exp(conf[1]),
        # p-values
        "pval": whichModel.pvalues
    })
    return out.sort_values(by="OR", ascending=False)

# Function Predicted probabilities for each involvement dummy (set one = 1, others = 0; hold rest at means)

# def predicted_probabilities(whichModel, dummy_cols):
#     # Get the mean values of the continuous variables
#     means = dfUser.mean()
#     # Create a DataFrame to hold the predicted probabilities
#     preds = pd.DataFrame(index=dummy_cols)
#     for col in dummy_cols:
#         # Set the current dummy variable to 1 and others to 0
#         X = pd.DataFrame(0, index=np.arange(1), columns=dummy_cols)
#         X[col] = 1
#         # Add the mean values of the continuous variables
#         for var in means.index:
#             if var not in dummy_cols:
#                 X[var] = means[var]
#         # Get the predicted probabilities
#         preds[col] = whichModel.predict(X)
#     return preds

# Predicted probability for each involvement dummy (set one=1, others=0; all other dummies=0)
def group_pred_by_singleton(whichModel, whichColumns, group_name="Group"):
    # create baseline scenario - set all dummy variables to 0 (reference categories). I will turn on what I want to test
    base = {col: 0 for col in whichModel.model.exog_names if col not in ["Intercept"]}
    # list to store prediction results
    results = []
    for g in whichColumns:
        # creates a copy to avoid modifying the original baseline
        scenario = base.copy()
        # Set only the current group dummy to 1, others stay 0, allowing me to loop through each dummy variable set it to 1 and then leave the others at 0
        scenario[g] = 1
        prob = whichModel.predict(pd.DataFrame([scenario]))[0]
        results.append({group_name: g, "PredProb_Severe": prob})
    return pd.DataFrame(results).sort_values("PredProb_Severe", ascending=False)

#Print and save Predicted probabilities for Models A and Model B
print("\n=== Predicted probability of severe injury by Peak/Off-Peak ===")
predProbPeak= group_pred_by_singleton(modelA, peakCols, group_name="PeakOffPeak")
display(predProbPeak)
predProbPeak.to_csv(os.path.join(destinationFolder, "TimeModelA_Stats.csv"),  mode='a', header=True,index=False)

print("\n=== Predicted probability of severe injury by Season ===")
predProb_Season = group_pred_by_singleton(modelB, seasonCols, group_name="Season")
display(predProb_Season)
predProb_Season.to_csv(os.path.join(destinationFolder, "TimeModelB_Stats.csv"), mode='a', header=True, index=False)

print("\n=== Predicted probability of severe injury by Month ===")
predProb_Month = group_pred_by_singleton(modelC, monthCols, group_name="Month")
display(predProb_Month)
predProb_Month.to_csv(os.path.join(destinationFolder, "TimeModelC_Stats.csv"), mode='a', header=True, index=False)

print("\n=== Predicted probability of severe injury by Hour ===")
predProb_Hour = group_pred_by_singleton(modelD, hourCols, group_name="Hour")
display(predProb_Hour)
predProb_Hour.to_csv(os.path.join(destinationFolder, "TimeModelD_Stats.csv"), mode='a', header=True, index=False)

print("\n=== Predicted probability of severe injury by TimeRange ===")
predProb_TimeRange = group_pred_by_singleton(modelE, timeRangeCols, group_name="TimeRange")
display(predProb_TimeRange)
predProb_TimeRange.to_csv(os.path.join(destinationFolder, "TimeModelE_Stats.csv"), mode='a', header=True, index=False)


# Print and save outputs of odds ratios
modelA_or_ci = odds_ratios_and_ci(modelA)
print("\n=== Odds Ratios for Model A (Peak) ===")
display(modelA_or_ci)
modelA_or_ci.to_csv(os.path.join(destinationFolder, "TimeModelA_Stats.csv"), mode='a', header=True, index=False)

modelB_or_ci = odds_ratios_and_ci(modelB)
print("\n=== Odds Ratios for Model B (Season) ===")
display(modelB_or_ci)
modelB_or_ci.to_csv(os.path.join(destinationFolder, "TimeModelB_Stats.csv"), mode='a', header=True, index=False)

modelC_or_ci = odds_ratios_and_ci(modelC)
print("\n=== Odds Ratios for Model C (Month) ===")
display(modelC_or_ci)
modelC_or_ci.to_csv(os.path.join(destinationFolder, "TimeModelC_Stats.csv"), mode='a', header=True, index=False)

modelD_or_ci = odds_ratios_and_ci(modelD)
print("\n=== Odds Ratios for Model D (Hour) ===")
display(modelD_or_ci)
modelD_or_ci.to_csv(os.path.join(destinationFolder, "TimeModelD_Stats.csv"), mode='a', header=True, index=False)

modelE_or_ci = odds_ratios_and_ci(modelE)
print("\n=== Odds Ratios for Model E (TimeRange) ===")
display(modelE_or_ci)
modelE_or_ci.to_csv(os.path.join(destinationFolder, "TimeModelE_Stats.csv"), mode='a', header=True, index=False)


# # wet-dark predicted probability
# # Fit logistic regression model

# # formulaC = 'SevereInjury ~ ' + ' + '.join(envLightCondCols)
# # modelC = smf.logit(formula=formulaC, data=dfEnv).fit()
# # print("\n Model C: Wet-DDark vs Severe Injuries \n")
# # print(modelC.summary())

# formulaC="""
#     SevereInjury ~ EnvLightingCondition_Dark 
#                    + EnvLightingCondition_Dark__artificial
#                    + EnvLightingCondition_Dusk
#                    + EnvLightingCondition_Dusk__artificial
#                    + EnvLightingCondition_Dawn
#                    + EnvLightingCondition_Dawn__artificial
#                    + EnvRoadSurfaceCondition_Wet
#                    + EnvRoadSurfaceCondition_Ice
#                    + EnvRoadSurfaceCondition_Packed_Snow
#                    + EnvRoadSurfaceCondition_Slush
#                    + EnvRoadSurfaceCondition_Loose_Snow
#                    + EnvRoadSurfaceCondition_Loose_Sand_or_Gravel
#                    + EnvRoadSurfaceCondition_Spilled_liquid
#                    + EnvRoadSurfaceCondition_Other
#     """
# modelC = smf.logit(
#     formula=formulaC,
#     data=dfEnv
# ).fit()

# # Create input for wet-dark conditions (baseline: Daylight, Dry)
# wet_dark = pd.DataFrame({
#     "EnvLightingCondition_Dark": [1],
#     "EnvLightingCondition_Dark__artificial": [0],
#     "EnvLightingCondition_Dawn": [0],
#     "EnvLightingCondition_Dawn__artificial": [0],
#     "EnvLightingCondition_Daylight__artificial": [0],
#     "EnvLightingCondition_Dusk": [0],
#     "EnvLightingCondition_Dusk__artificial": [0],
#     "EnvLightingCondition_Other": [0],
#     "EnvRoadSurfaceCondition_Ice": [0],
#     "EnvRoadSurfaceCondition_Loose_Sand_or_Gravel": [0],
#     "EnvRoadSurfaceCondition_Loose_Snow": [0],
#     "EnvRoadSurfaceCondition_Other": [0],
#     "EnvRoadSurfaceCondition_Packed_Snow": [0],
#     "EnvRoadSurfaceCondition_Slush": [0],
#     "EnvRoadSurfaceCondition_Spilled_liquid": [0],
#     "EnvRoadSurfaceCondition_Wet": [1]
# })

# # Predict probability for Wet–Dark
# pred_prob_wet_dark = modelC.predict(wet_dark)
# print("\nPredicted Probability of Severe Collision (Wet–Dark):", float(pred_prob_wet_dark))
# pred_prob_wet_dark.to_csv(os.path.join(destinationFolder, "EnvModelC_Stats.csv"),  mode='a', header=True,index=False)

# # Find the deadliest coobminations of lighting and road surface conditions

# # #  Generate all combinations of lighting × road surface conditions
# # lightingCols = [
# #     "EnvLightingCondition_Dark",
# #     "EnvLightingCondition_Dark__artificial",
# #     "EnvLightingCondition_Dusk",
# #     "EnvLightingCondition_Dusk__artificial",
# #     "EnvLightingCondition_Dawn",
# #     "EnvLightingCondition_Dawn__artificial"
# # ]

# # road_vars = [
# #     "EnvRoadSurfaceCondition_Wet",
# #     "EnvRoadSurfaceCondition_Ice",
# #     "EnvRoadSurfaceCondition_Packed_Snow",
# #     "EnvRoadSurfaceCondition_Slush",
# #     "EnvRoadSurfaceCondition_Loose_Snow",
# #     "EnvRoadSurfaceCondition_Loose_Sand_or_Gravel",
# #     "EnvRoadSurfaceCondition_Spilled_liquid",
# #     "EnvRoadSurfaceCondition_Other"
# # ]

# combinations = list(product(envLightCondCols, envRoadSurfCols))

# # Create DataFrame for each combination and predict probabilities
# results = []

# for light, road in combinations:
#     # Set all dummy vars = 0
#     row = {v: 0 for v in envLightCondCols + envRoadSurfCols}
    
#     # Turn on this combination
#     row[light] = 1
#     row[road] = 1
    
#     # Predict probability
#     prob = modelC.predict(pd.DataFrame([row]))[0]
    
#     results.append({
#         "LightingCondition": light,
#         "RoadSurfaceCondition": road,
#         "PredictedProbability": prob
#     })

# # Create a results DataFrame and find the max
# pred_df = pd.DataFrame(results)
# most_dangerous = pred_df.sort_values("PredictedProbability", ascending=False).head(1)
# print("Most Dangerous Combination:\n", most_dangerous)
# most_dangerous.to_csv(os.path.join(destinationFolder, "EnvModelC_Stats.csv"),  mode='a', header=True,index=False)



Time peak/off-peak dummies: ['TimePeakOffPeak_OFF_PEAK'] ...total: 1
Time season dummies: ['TimeSeason_WINTER'] ...total: 1
Time range dummies: ['TimeRange_15_00_to_17_59', 'TimeRange_18_00_to_20_59', 'TimeRange_21_00_to_23_59', 'TimeRange_3_00_to_5_59', 'TimeRange_6_00_to_8_59'] ...total: 7
Time month dummies: ['TimeMonth_April', 'TimeMonth_August', 'TimeMonth_December', 'TimeMonth_February', 'TimeMonth_July'] ...total: 11
Time hour dummies: ['TimeHour_0', 'TimeHour_1', 'TimeHour_2', 'TimeHour_3', 'TimeHour_4'] ...total: 23
Optimization terminated successfully.
         Current function value: 0.574771
         Iterations 5

 Model A: Time Peak/Off-Peak vs Severe Injuries 

                           Logit Regression Results                           
Dep. Variable:           SevereInjury   No. Observations:                10044
Model:                          Logit   Df Residuals:                    10042
Method:                           MLE   Df Model:                            1


,PeakOffPeak,PredProb_Severe
0,TimePeakOffPeak_OFF_PEAK,0.721207



=== Predicted probability of severe injury by Season ===


,Season,PredProb_Severe
0,TimeSeason_WINTER,0.731825



=== Predicted probability of severe injury by Month ===


...

,OR,2.5%,97.5%,pval
Intercept,2.470588,2.099740,2.906934,1.160612e-27
TimeMonth_November,1.472693,1.167021,1.858429,1.109310e-03
TimeMonth_October,1.361130,1.089294,1.700802,6.679388e-03
TimeMonth_December,1.220643,0.969139,1.537416,9.032713e-02
TimeMonth_September,1.206286,0.971779,1.497385,8.905311e-02
TimeMonth_March,1.142173,0.900814,1.448201,2.724078e-01
TimeMonth_June,1.123828,0.906373,1.393454,2.873308e-01
TimeMonth_February,1.099353,0.862463,1.401309,4.442763e-01
TimeMonth_July,1.088187,0.876654,1.350761,4.434915e-01
TimeMonth_August,1.078885,0.872147,1.334629,4.841960e-01



=== Odds Ratios for Model D (Hour) ===


,OR,2.5%,97.5%,pval
Intercept,2.699187,2.194741,3.319576,5.112989e-21
TimeHour_7,1.439413,1.015084,2.041122,4.095557e-02
TimeHour_8,1.361771,0.979104,1.893999,6.657924e-02
TimeHour_18,1.327995,1.005474,1.753970,4.567104e-02
TimeHour_9,1.257141,0.918236,1.721130,1.533607e-01
TimeHour_15,1.249300,0.939184,1.661815,1.262739e-01
TimeHour_6,1.238323,0.887221,1.728368,2.089163e-01
TimeHour_19,1.232068,0.926029,1.639248,1.520097e-01
TimeHour_11,1.176825,0.868471,1.594662,2.935853e-01
TimeHour_17,1.147299,0.871514,1.510354,3.272942e-01



=== Odds Ratios for Model E (TimeRange) ===


,OR,2.5%,97.5%,pval
Intercept,2.498824,2.232945,2.796360,2.607287e-57
TimeRange_6_00_to_8_59,1.448759,1.197667,1.752491,1.348602e-04
TimeRange_9_00_to_11_59,1.259852,1.060416,1.496797,8.610897e-03
TimeRange_18_00_to_20_59,1.229937,1.052292,1.437570,9.311718e-03
TimeRange_15_00_to_17_59,1.176554,1.007388,1.374127,4.008245e-02
TimeRange_21_00_to_23_59,1.053469,0.893888,1.241538,5.342633e-01
TimeRange_Midnight_to_2_59,0.924071,0.771262,1.107156,3.918764e-01
TimeRange_3_00_to_5_59,0.882701,0.714724,1.090157,2.466716e-01
